# Confound-control method validation — is the established result trustworthy?
One patient (PTYEU_task147), hippocampus/self. Three alternative confound-control methods were tried against the established xcirc/variance-partitioning approach: a confound-matched word-substitution null, plain (Pearson-residual) residualization, and a more careful IRLS-weighted residualization. All three disagreed sharply with xcirc on real data. To settle which method is actually trustworthy, we built simulated datasets with a KNOWN ground truth (zero true effect in one direction, real injected effect in the other) and checked which methods' raw p-values are correctly Uniform(0,1) under the true null.

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

VP_DIR  = '/scratch/aniluchavez/ConvoDATAS/VPResults'
FIG_DIR = '../figures'

vp_main = pickle.load(open(f'{VP_DIR}/gpt2-xl_ctx200_worddur_xcirc/pc100/PTYEU_task147_L36_VP.pkl', 'rb'))
vp_sub  = vp_main[(vp_main['region']=='hippocampus') & (vp_main['condition']=='self')]
n = len(vp_sub)

cm = pickle.load(open(f'{VP_DIR}/confound_matched_perm_prototype_PTYEU_hippo_self.pkl', 'rb'))
pr = pickle.load(open(f'{VP_DIR}/residualization_prototype_PTYEU_hippo_self.pkl', 'rb'))
ir = pickle.load(open(f'{VP_DIR}/residualization_irls_prototype_PTYEU_hippo_self.pkl', 'rb'))
cal1 = pickle.load(open(f'{VP_DIR}/null_calibration_check_PTYEU_hippo_self.pkl', 'rb')).iloc[0]
cal2 = pickle.load(open(f'{VP_DIR}/null_calibration_check_symmetric_PTYEU_hippo_self.pkl', 'rb')).iloc[0]
print(f'n_neurons={n}')

## Panel 1 — on REAL data, the methods sharply disagree

In [ ]:
real_data_pct = {
    'xcirc\n(established)':           100 * vp_sub['significant'].sum() / n,
    'matched\nsubstitution\n(k=50)':  100 * cm['significant_confmatch_c50'].sum() / n,
    'plain\nresidualization':         100 * pr['significant_resid'].sum() / n,
    'IRLS\nresidualization':          100 * ir['significant_resid_irls'].sum() / n,
}

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#2166ac', '#d6604d', '#d6604d', '#d6604d']
bars = ax.bar(real_data_pct.keys(), real_data_pct.values(), color=colors)
for b, v in zip(bars, real_data_pct.values()):
    ax.text(b.get_x()+b.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontsize=10)
ax.set_ylabel('% significant neurons (real data)')
ax.set_title('PTYEU_task147, hippocampus/self — methods disagree sharply on real data')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/09_real_data_disagreement.pdf', bbox_inches='tight')
plt.show()

## Panel 2 — on data with a KNOWN zero effect, only xcirc is calibrated
`null_calibration_check.py`: simulated `Y_sim ~ Poisson(controls-only \u03bb\u0302)` — zero true semantic effect by construction. The correct calibration metric is the % of RAW (uncorrected) p-values below 0.05, which should be ~5% for a well-calibrated test (FDR-corrected counts are not the right metric here — under a true global null, BH-FDR controls the chance of *any* false rejection at \u22645%, so near-0% FDR-corrected hits is expected, not necessarily conservative).

In [ ]:
def pct_below(pv, thresh=0.05):
    pv = np.asarray(pv); pv = pv[~np.isnan(pv)]
    return 100 * np.mean(pv < thresh)

calib_pct = {
    'xcirc\nunique_semantic':        pct_below(cal1['p_vals_xcirc']),
    'plain\nresidualization':        pct_below(cal1['p_vals_plain_resid']),
    'IRLS\nresidualization':         pct_below(cal1['p_vals_irls_resid']),
    'xcirc\nunique_controls\n(corrected null)': pct_below(cal2['p_vals_unique_controls_calibration']),
}

fig, ax = plt.subplots(figsize=(7.5, 5))
colors = ['#2166ac', '#d6604d', '#d6604d', '#2166ac']
bars = ax.bar(calib_pct.keys(), calib_pct.values(), color=colors)
for b, v in zip(bars, calib_pct.values()):
    ax.text(b.get_x()+b.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontsize=10)
ax.axhline(5, color='k', linestyle='--', linewidth=1.2, label='nominal 5% (well-calibrated)')
ax.set_ylabel('% raw p-values < 0.05 (on known-zero-effect data)')
ax.set_title('Calibration check: only xcirc stays near the nominal rate')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/09_calibration_check_bars.pdf', bbox_inches='tight')
plt.show()

## Panel 3 — raw p-value histograms (the actual calibration signature)
A well-calibrated test's raw p-values under a true null look flat/Uniform(0,1). A method that's anti-conservative (inflates false positives) shows a left-skewed histogram (too many small p-values).

In [ ]:
panels = [
    ('xcirc — unique_semantic\n(zero true effect)', cal1['p_vals_xcirc'], '#2166ac'),
    ('xcirc — unique_controls\n(zero true effect, corrected)', cal2['p_vals_unique_controls_calibration'], '#2166ac'),
    ('plain residualization\n(zero true effect)', cal1['p_vals_plain_resid'], '#d6604d'),
    ('IRLS residualization\n(zero true effect)', cal1['p_vals_irls_resid'], '#d6604d'),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (title, pv, color) in zip(axes, panels):
    pv = np.asarray(pv); pv = pv[~np.isnan(pv)]
    ax.hist(pv, bins=10, range=(0,1), color=color, edgecolor='white')
    ax.axhline(len(pv)/10, color='k', linestyle='--', linewidth=1, label='uniform expectation')
    ax.set_title(title, fontsize=9.5)
    ax.set_xlabel('raw p-value')
axes[0].set_ylabel('neuron count')
axes[0].legend(frameon=False, fontsize=8)
plt.suptitle('Raw p-value distributions under a KNOWN zero-effect null — flat = calibrated, left-skewed = inflated false positives', y=1.05)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/09_pvalue_histograms.pdf', bbox_inches='tight')
plt.show()

## Panel 4 — power check: xcirc also correctly detects a REAL injected effect
Same `unique_semantic` test, but now on `Y_sim2 ~ Poisson(\u03bb\u0302_sem)` — a dataset with a real injected semantic effect (and zero true controls effect). A useful test isn't just conservative; it should also fire when there's something real to find.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

ax = axes[0]
vals = [cal2['unique_semantic_power'], cal2['unique_controls_calibration']]
labels = ['unique_semantic\n(real effect present)', 'unique_controls\n(zero true effect)']
ax.bar(labels, vals, color=['#2166ac', '#92c5de'])
for i, v in enumerate(vals):
    ax.text(i, v+1, f'{v:.1f}%', ha='center')
ax.axhline(5, color='k', linestyle='--', linewidth=1, label='nominal 5%')
ax.set_ylabel('% significant (FDR-corrected)')
ax.set_title('xcirc on Y_sim2: power vs. calibration')
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
pv = np.asarray(cal2['p_vals_unique_semantic_power']); pv = pv[~np.isnan(pv)]
ax.hist(pv, bins=10, range=(0,1), color='#2166ac', edgecolor='white')
ax.axhline(len(pv)/10, color='k', linestyle='--', linewidth=1)
ax.set_xlabel('raw p-value')
ax.set_title('unique_semantic raw p-values\n(real effect -- should be left-skewed)')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/09_power_check.pdf', bbox_inches='tight')
plt.show()